> 📦 **Note:** Before running this notebook, make sure **BodoSQL and related dependencies** are installed.

You can install it using the following command

- If you're working inside the repo:
  ```bash
  pip install -e ".[bodosql]"
  ```

- Or install BodoSQL directly:
  ```bash
  pip install bodosql
  ```

### 📦 Importing Required Libraries

In [1]:
import pydough
import bodosql
import datetime
import pandas as pd

/Users/kiannassre/Desktop/PyDough/.venv/lib/python3.12/site-packages/bodosql/py4j_gateway.py:46: UserWarning: $JAVA_HOME is currently unset. This occurs when OpenJDK is not installed in your conda environment or when your environment has recently changed by not reactivates. BodoSQL will default to using you system's Java.It is recommended that you use OpenJDK v17 from Conda with BodoSQL. To do so, first run
    conda install openjdk=17 -c conda-forge
and then reactivate your environment via
    conda deactivate && conda activate /Users/kiannassre/miniforge3
  warnings.warn(


### 🔐 BodoSQLContext

BodoSQL is a versatile SQL compute engine that can work with data from various sources.
The [BodoSQL API](https://docs.bodo.ai/latest/api_docs/sql/database_catalogs/)
defines several ways to connect a BodoSQL context object to a data source, such
as a Snowflake account. This demo shows how to run PyDough without any catalog, instead just having
a suite of tables defined to BodoSQL via a dictionary mapping table names to Pandas DataFrames.

1. **Create all of the pd.DataFrame tables we wish to use**:
   - [See here](bodosql_color_tables.py) for the script that creates the 4 tables used by this example (CLRS, SHPMNTS, CUST, SUPLS)

2. **Build the BodoSQLContext connected to the local tables**:
   - The first argument is a dictionary mapping each table name to the Pandas DataFrame. These are all of the tables that BodoSQL will have access to.

3. **Connect to BodoSQL using PyDough**:
   - `pydough.active_session.load_metadata_graph(...)` loads a metadata graph that maps your Snowflake schema (used for query planning or optimizations).
   - `connect_database(...)` uses the loaded credentials to establish a live connection to your Snowflake database.

📌 Make sure:
- The metadata graph path points to a valid JSON file that represents your schema.


In [2]:
import json
from bodosql_color_tables import generate_color_tables

# Step 1: Generate the four tables being used
color_df, customers_df, suppliers_df, shipments_df = generate_color_tables()

# Step 2: Build the BodoSQLContext connected to the four tables
bc = bodosql.BodoSQLContext({
    "CLRS": color_df,
    "SHPMNTS": shipments_df,
    "CUST": customers_df,
    "SUPLS": suppliers_df,
})

# Step 3: Load a sample metadata graph and connect PyDough to the BodoSQL context
pydough.active_session.load_metadata_graph("../metadata/bodosql_demo_graph.json", "COLORSHOP")
pydough.active_session.connect_database("bodosql", context=bc)

DatabaseContext(connection=<bodosql.context.BodoSQLContext object at 0x12dfb56a0>, dialect=<DatabaseDialect.SNOWFLAKE: 'snowflake'>)

### 🔌 Enabling PyDough's Jupyter Magic Commands

This line loads the `pydough.jupyter_extensions` module, which adds **custom magic commands** (like `%%pydough`) to the notebook.

These magic commands allow you to:
- Write PyDough directly in notebook cells using `%%pydough`
- Automatically render results

This is a Jupyter-specific feature — the `%load_ext` command dynamically loads these extensions into your current notebook session.


In [3]:
%load_ext pydough.jupyter_extensions

### 📊 Basic Query Using PyDough DSL

This cell runs a simple query on these tables using PyDough's Python-style DSL instead of raw SQL.

The query computes the total volume of each paint color purchased by customer Quentin Sharma, finding the 3 paint colors purhcased the most by that customer.

Finally, `pydough.to_df(output)` converts and prints the result as a Pandas DataFrame for easy inspection and analysis in Python.


In [4]:
%%pydough

selected_shipments = shipments.WHERE(
    (customer.first_name == "Quentin") &
    (customer.last_name == "Sharma")
)

output = (
    colors
    .CALCULATE(name, total_quantity=SUM(selected_shipments.volume))
    .TOP_K(3, by=total_quantity.DESC())
)

# Step 3: Execute code
pydough.to_df(output)

,NAME,TOTAL_QUANTITY
0,Flamingo Pink,66.5
1,Beau Blue,65.0
2,Dark Powder Blue,62.5


Next, use the same context to execute SQL generated from PyDough, and also to
see the execution plan generated by BodoSQL.

In [5]:
%%pydough

as_sql = pydough.to_sql(output)
print("Generated SQL:\n", as_sql)

plan = bc.generate_plan(as_sql)
print("\nBodoSQL Query plan:\n", plan)

bc.sql(as_sql)

Generated SQL:
 WITH _s3 AS (
  SELECT
    shpmnts.colid,
    SUM(shpmnts.vol) AS sum_vol
  FROM shpmnts AS shpmnts
  JOIN cust AS cust
    ON cust.cstfname = 'Quentin'
    AND cust.cstid = shpmnts.cusid
    AND cust.cstlname = 'Sharma'
  GROUP BY
    1
)
SELECT
  clrs.colorname AS name,
  COALESCE(_s3.sum_vol, 0) AS total_quantity
FROM clrs AS clrs
LEFT JOIN _s3 AS _s3
  ON _s3.colid = clrs.identname
ORDER BY
  2 DESC NULLS LAST
LIMIT 3

BodoSQL Query plan:
 CombineStreamsExchange
  BodoPhysicalSort(sort0=[$1], dir0=[DESC-nulls-last], fetch=[3])
    BodoPhysicalProject(NAME=[$1], TOTAL_QUANTITY=[COALESCE($3, 0.0E0:DOUBLE)])
      BodoPhysicalJoin(condition=[=($2, $0)], joinType=[left])
        SeparateStreamExchange
          PandasToBodoPhysicalConverter
            PandasProject(IDENTNAME=[$0], COLORNAME=[$1])
              PandasTableScan(table=[[__BODOLOCAL__, CLRS]])
        BodoPhysicalAggregate(group=[{0}], SUM_VOL=[SUM($1)])
          BodoPhysicalProject(COLID=[$0], $f5=[*($2,

,NAME,TOTAL_QUANTITY
0,Flamingo Pink,66.5
1,Beau Blue,65.0
2,Dark Powder Blue,62.5
